In [57]:
import networkx as nx
import matplotlib.pyplot as plt
import json
import MeCab
import unicodedata
import time
import subprocess
import re
import subprocess
from collections import Counter
from tqdm import tqdm
file_path = "jpn_news_2023_30K-sentences.txt"
lines = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        cleaned_line = line.strip()
        
        text = re.sub(r"^\d+\s*", "", cleaned_line)

        lines.append(text)

lines = lines[1:500]

In [17]:
with open("udpipe_tokens.json", "r", encoding="utf-8") as f:
    data = json.load(f)

all_tokens_udpipe = data["all_tokens"]
clean_tokens_udpipe = data["clean_tokens"]
all_pos_udpipe = data['pos']

In [3]:
command_jumanpp = 'docker run -i --rm --platform linux/amd64 kunlp/jumanpp-knp jumanpp'
command_knp = 'docker run -i --rm --platform linux/amd64 kunlp/jumanpp-knp knp -anaphora'
tagger = MeCab.Tagger()

In [4]:
process_jumanpp = subprocess.Popen(command_jumanpp, shell=True, stdin=subprocess.PIPE, stdout=subprocess.PIPE)
juman_output = process_jumanpp.communicate(input=text.encode('utf-8'))[0]

In [5]:
def run_juman(text):
    command = 'docker run -i --rm --platform linux/amd64 kunlp/jumanpp-knp jumanpp'

    process = subprocess.Popen(
        command,
        shell=True,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    output, error = process.communicate(input=text.encode('utf-8'))

    if error:
        print(error.decode())

    return output.decode('utf-8')

In [6]:
def avg_token_length(tokens):
    return sum(len(t) for t in tokens) / len(tokens)

In [8]:
def get_boundaries(tokens):
    boundaries = set()
    pos = 0
    for tok in tokens[:-1]:
        pos += len(tok)
        boundaries.add(pos)
    return boundaries

In [9]:
def boundary_metrics(tokens1, tokens2):
    b1 = get_boundaries(tokens1)
    b2 = get_boundaries(tokens2)
    
    tp = len(b1 & b2) 
    fp = len(b2 - b1)
    fn = len(b1 - b2)
    
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0)
    
    return precision, recall, f1

In [10]:
def token_agreement(tokens1, tokens2):
    set1 = set(tokens1)
    set2 = set(tokens2)
    
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    
    return intersection / union if union != 0 else 0

In [12]:
results = []

start = time.perf_counter()

for line in lines:
    result = run_juman(line).split('\n')
    results.append(result)

end = time.perf_counter()

print(end - start)

140.31835170800332


In [14]:
b = []
for result in results:
    for i in result:
        a = i.split('"')
        b.append(a)

s = []
for i in b: 
    s.append(i[0].split())

In [26]:
all_tokens_juman = []
clean_tokens_juman = []
all_pos_juman = []

for line in lines:
    result = run_juman(line).split('\n')
    
    for r in result:
        if r.strip() == 'EOS' or not r.strip():
            continue
        
        parts = r.split()
        if len(parts) < 4:
            continue
        
        surface = parts[0]
        if surface == '@':
                continue
        pos = parts[3]   # вот это и есть POS
        
        norm = unicodedata.normalize('NFKC', surface)
        
        all_tokens_juman.append(norm)
        all_pos_juman.append(pos)
        
        if norm.isdigit():
            continue
        
        if not any(unicodedata.category(c).startswith('L') for c in norm):
            continue
        
        clean_tokens_juman.append(norm)

In [12]:
print(len(clean_tokens_juman))

10380


In [13]:
print(round(avg_token_length(clean_tokens_juman),2))

1.79


In [33]:
start = time.perf_counter()
for line in lines:
    node = tagger.parseToNode(line)

end = time.perf_counter()

print(end - start)

0.017790166952181607


In [15]:
all_tokens_mecab = []
clean_tokens_mecab = []
all_pos_mecab = []             

for line in lines:
    node = tagger.parseToNode(line)
    
    while node:
        surface = node.surface
        
        if surface:
            feature = node.feature.split(',')   
            if len(feature) < 1:
                node = node.next
                continue

            pos = feature[0]                    

            norm = unicodedata.normalize('NFKC', surface)
            
            all_tokens_mecab.append(norm)
            all_pos_mecab.append(pos)         
            
            if norm.isdigit():
                node = node.next
                continue

            if not any(unicodedata.category(c).startswith('L') for c in norm):
                node = node.next
                continue

            clean_tokens_mecab.append(norm)

        node = node.next

In [42]:
print(len(clean_tokens_mecab))

11118


In [43]:
print(round(avg_token_length(clean_tokens_mecab),2))

1.66


In [48]:
tools = {
    "juman": all_tokens_juman,
    "mecab": all_tokens_mecab,
    "udpipe": all_tokens_udpipe
}

for t1 in tools:
    for t2 in tools:
        p, r, f1 = boundary_metrics(tools[t1], tools[t2])
        print(f"{t1} vs {t2}: F1 = {f1:.3f}")

juman vs juman: F1 = 1.000
juman vs mecab: F1 = 0.605
juman vs udpipe: F1 = 0.604
mecab vs juman: F1 = 0.605
mecab vs mecab: F1 = 1.000
mecab vs udpipe: F1 = 0.633
udpipe vs juman: F1 = 0.604
udpipe vs mecab: F1 = 0.633
udpipe vs udpipe: F1 = 1.000


In [50]:
systems = {
    "juman": all_tokens_juman,
    "mecab": all_tokens,
    "udpipe": all_tokens_udpipe
}

for s1 in systems:
    for s2 in systems:
        score = token_agreement(systems[s1], systems[s2])
        print(f"{s1} vs {s2}: {score:.3f}")

juman vs juman: 1.000
juman vs mecab: 0.724
juman vs udpipe: 0.612
mecab vs juman: 0.724
mecab vs mecab: 1.000
mecab vs udpipe: 0.720
udpipe vs juman: 0.612
udpipe vs mecab: 0.720
udpipe vs udpipe: 1.000


In [55]:
set_juman = set(all_tokens_juman)
set_udpipe = set(all_tokens_udpipe)
set_mecab = set(all_tokens_mecab)


only_juman = set_juman - set_udpipe
only_udpipe = set_udpipe - set_mecab

only_mecab =  set_mecab - set_juman

In [27]:
import re

target_verbs = [
    "続けている",
    "栽培される",
    "発生させ",
    "始まった",
    "もらいたい"
]

def normalize(tok):
    return re.sub(r"\s+", "", tok)

def find_forms(tokens, pos_tags, targets, window=3, max_span=8):
    out = []
    n = len(tokens)
    norm_tokens = [normalize(t) for t in tokens]

    for form in targets:
        found = False

        for i in range(n):
            for j in range(i + 1, min(n, i + max_span) + 1):
                chunk = "".join(norm_tokens[i:j])
                if chunk == form:
                    out.append({
                        "form": form,
                        "tokens": tokens[i:j],
                        "pos": pos_tags[i:j],
                        "ctx_tokens": tokens[max(0, i-window):i] + [f"[{t}]" for t in tokens[i:j]] + tokens[j:j+window],
                        "ctx_pos": pos_tags[max(0, i-window):i] + [f"[{p}]" for p in pos_tags[i:j]] + pos_tags[j:j+window],
                    })
                    found = True
                    break
            if found:
                break

        if not found:
            out.append({
                "form": form,
                "tokens": None,
                "pos": None,
                "ctx_tokens": None,
                "ctx_pos": None
            })

    return out


systems = {
    "JUMAN": (all_tokens_juman, all_pos_juman),
    "MeCab": (all_tokens_mecab, all_pos_mecab),
    "UDPipe": (all_tokens_udpipe, all_pos_udpipe),
}

for sys_name, (tokens, pos_tags) in systems.items():
    print("\n" + "=" * 60)
    print(sys_name)
    print("=" * 60)

    rows = find_forms(tokens, pos_tags, target_verbs)

    for r in rows:
        print(f"\n{r['form']}")
        if r["tokens"] is None:
            print("  не найдено")
        else:
            print("  tokens:", " | ".join(r["tokens"]))
            print("  pos   :", " | ".join(r["pos"]))
            print("  ctx   :", " ".join(r["ctx_tokens"]))
            print("  posctx:", " ".join(r["ctx_pos"]))


JUMAN

続けている
  tokens: 続けて | いる
  pos   : 動詞 | 接尾辞
  ctx   : 相互 派遣 を [続けて] [いる] 。 13 年度
  posctx: 名詞 名詞 助詞 [動詞] [接尾辞] 特殊 名詞 接尾辞

栽培される
  tokens: 栽培 | さ | れる
  pos   : 名詞 | 動詞 | 接尾辞
  ctx   : 地域 など で [栽培] [さ] [れる] マンゴー 、 モンキー
  posctx: 名詞 助詞 助詞 [名詞] [動詞] [接尾辞] 未定義語 特殊 名詞

発生させ
  tokens: 発生 | さ | せ
  pos   : 名詞 | 動詞 | 接尾辞
  ctx   : で 地熱 を [発生] [さ] [せ] 、 熱帯 地域
  posctx: 助詞 名詞 助詞 [名詞] [動詞] [接尾辞] 特殊 名詞 名詞

始まった
  tokens: 始まった
  pos   : 動詞
  ctx   : 2014 年 から [始まった] 助成 事業 。
  posctx: 名詞 接尾辞 助詞 [動詞] 名詞 名詞 特殊

もらいたい
  tokens: もらい | たい
  pos   : 接尾辞 | 接尾辞
  ctx   : 賑わい を 感じて [もらい] [たい] 」 と して
  posctx: 名詞 助詞 動詞 [接尾辞] [接尾辞] 特殊 助詞 動詞

MeCab

続けている
  tokens: 続け | て | いる
  pos   : 動詞 | 助詞 | 動詞
  ctx   : 相互 派遣 を [続け] [て] [いる] 。 13 年度
  posctx: 名詞 名詞 助詞 [動詞] [助詞] [動詞] 補助記号 名詞 名詞

栽培される
  tokens: 栽培 | さ | れる
  pos   : 名詞 | 動詞 | 助動詞
  ctx   : 地域 など で [栽培] [さ] [れる] マンゴー 、 モンキー
  posctx: 名詞 助詞 助詞 [名詞] [動詞] [助動詞] 名詞 補助記号 名詞

発生させ
  tokens: 発生 | さ | せ
  pos   : 名詞 | 動詞 | 助動詞
  ctx   : で 地熱 を [発生] [さ] [せ] 

In [31]:
def run_knp(text):
    command = 'docker run -i --rm --platform linux/amd64 kunlp/jumanpp-knp bash -c "jumanpp | knp -tab"'

    process = subprocess.Popen(
        command,
        shell=True,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    output, error = process.communicate(input=text.encode('utf-8'))

    if error:
        print(error.decode())

    return output.decode('utf-8')

In [32]:
text = "\n".join(lines)

In [33]:
knp_result = run_knp(text)

In [50]:
def count_dependencies(knp_output):
    return sum(
        1
        for line in knp_output.splitlines()
        if line.startswith("*") and "D" in line and "-1D" not in line
    )

def split_sentences(knp_output):
    sentences = []
    current = []

    for line in knp_output.splitlines():
        current.append(line)
        if line.strip() == "EOS":
            sentences.append(current)
            current = []

    return sentences


def average_dependencies(knp_output):
    sentences = split_sentences(knp_output)

    counts = []

    for sent in sentences:
        deps = count_dependencies("\n".join(sent))
        counts.append(deps)

    return sum(counts) / len(counts) if counts else 0

def sentence_with_max_dependencies(knp_output):
    sentences = split_sentences(knp_output)

    max_count = -1
    max_sentence = None
    max_index = -1

    for i, sent in enumerate(sentences):
        deps = count_dependencies("\n".join(sent))

        if deps > max_count:
            max_count = deps
            max_sentence = sent
            max_index = i

    return max_index, max_count, max_sentence

In [35]:
count_dependencies(knp_result)

3843

In [37]:
average_dependencies(knp_result)

7.701402805611222

In [66]:
def run_cabocha(text):
    command = 'docker run -i --rm --platform linux/amd64 kttyo/python-cabocha:1.0 cabocha'

    process = subprocess.Popen(
        command,
        shell=True,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    output, error = process.communicate(input=text.encode('utf-8'))

    if error:
        err = error.decode('utf-8', errors='ignore')
        if err.strip():
            print(err)

    return output.decode('utf-8', errors='ignore')


cabocha_result = run_cabocha(text)

In [67]:
cabocha_result = run_cabocha(text)

In [68]:
print(cabocha_result)

morph.cpp(186) [charset() == decode_charset(dinfo->charset)] Incompatible charset: MeCab charset is EUC-JP, Your charset is UTF8



In [53]:
idx, count, sentence = sentence_with_max_dependencies(knp_result)

print(idx)
print(count)
print(lines[384])

384
41
２８２㌧と不漁だった昨年さえ５７％も下回ってしまった◆地球温暖化により海水温が高く、適温が１８度といわれるサケも岸に寄り付かず、それとは逆に夏の魚と言われ、どちらと言えば高めの水温好むオオナゴも不漁というのはどういうことなのか◆海の中が変わっているのは間違いなく南洋魚のフグやブリがよく獲れ漁師を悩ましている。


In [65]:
max_tokens = 0
max_sentence = None
current_tokens = []
current_sentence = []

for line in lines:
    result = run_juman(line).split('\n')

    for r in result:
        if r.strip() == 'EOS':
            if len(current_tokens) > max_tokens:
                max_tokens = len(current_tokens)
                max_sentence = current_sentence.copy()

            current_tokens = []
            current_sentence = []
            continue

        if not r.strip():
            continue

        parts = r.split()
        if len(parts) < 4:
            continue

        surface = parts[0]
        if surface == '@':
            continue

        norm = unicodedata.normalize('NFKC', surface)
        current_sentence.append(norm)

        if norm.isdigit():
            continue

        if not any(unicodedata.category(c).startswith('L') for c in norm):
            continue

        current_tokens.append(norm)

print( max_tokens)
print(" ".join(max_sentence))

Максимум токенов: 83
Предложение:
282 トン と 不漁 だった 昨年 さえ 57 % も 下回って しまった ◆ 地球 温暖 化 に より 海 水温 が 高く 、 適温 が 18 度 と いわ れる サケ も 岸 に 寄り付か ず 、 それ と は 逆に 夏 の 魚 と 言わ れ 、 どちら と 言えば 高 めの 水温 好む オオナゴ も 不漁 と いう の は どういう こと な の か ◆ 海 の 中 が 変わって いる の は 間違い なく 南洋 魚 の フグ や ブリ が よく 獲れ 漁師 を 悩ま して いる 。


In [54]:
sample = lines[72]

In [60]:
def run_knp_tree(text):
    command = 'docker run -i --rm --platform linux/amd64 kunlp/jumanpp-knp bash -c "jumanpp | knp "'

    process = subprocess.Popen(
        command,
        shell=True,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    output, error = process.communicate(input=text.encode('utf-8'))

    if error:
        print(error.decode())

    return output.decode('utf-8')

In [61]:
i = run_knp_tree(sample)

In [62]:
print(i)

# S-ID:1 KNP:5.0-bc4cef1 DATE:2026/04/29 SCORE:-98.03395
１２年──┐　　　　　　　　　　　　　　　　　　　　　　　　　<体言>
          余に──┐　　　　　　　　　　　　　　　　　　　　　<体言>
                わたる──┐　　　　　　　　　　　　　　　　　<用言:動><格解析結果:ガ/監禁;ニ/余>
                  拉致──┤　　　　　　　　　　　　　　　　　<体言>
                      監禁から──┐　　　　　　　　　　　　　<体言><格解析結果:ノ/->
                            解放された──┐　　　　　　　　　<用言:動><格解析結果:ガ/-;ニ/-;カラ/監禁;時間/-;外の関係/後>
                                          後、──┐　　　　　<体言>
                    後藤さんが──┐　　　　　　　│　　　　　<体言>
                            駆け込んだ──┐　　　│　　　　　<用言:動><格解析結果:ガ/後藤;ニ/-;デ/-;時間/交番>
                                          のは──┤　　　　　
                  マンションの──┐　　　　　　　│　　　　　<体言>
                         目と<P>─┤　　　　　　　│　　　　　<体言>
                         鼻の<P>─PARA──┐　　　│　　　　　<体言>
                                          先に──┤　　　　　<体言>
                                                  ある──┐　<用言:動><格解析結果:ガ/の;ニ/先;ト/-;時間/交番>
                                                  交番だった。<体言><用言:判><格解析結果:ノ/->
EOS



In [63]:
i = run_knp(sample)

In [64]:
count_dependencies(i)

12